### Nội Suy Newton Tiến Lùi (Tính Đạo Hàm)


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display, Math, Markdown
import math

def Newton_Interpolation(X_in, Y_in, t_val, degree=None):
    X = np.array(X_in, dtype=float)
    Y = np.array(Y_in, dtype=float)
    n = len(X) - 1
    if degree is None: degree = n
    else: degree = min(degree, n)
    
    h = X[1] - X[0]
    
    # Bảng sai phân
    diff_table = np.zeros((n + 1, n + 1))
    diff_table[:, 0] = Y
    for j in range(1, n + 1):
        for i in range(n - j + 1):
            diff_table[i][j] = diff_table[i+1][j-1] - diff_table[i][j-1]
            
    # Hiển thị bảng sai phân
    md = ["## ❖ BẢNG SAI PHÂN TỶ ĐỐI CÁCH ĐỀU"]
    header = "| $i$ | $x_i$ | $y_i$ | " + " | ".join([f"$\\Delta^{j} y$" for j in range(1, n+1)]) + " |"
    md.append(header)
    md.append("|---" * (n + 3) + "|")
    
    for i in range(n + 1):
        row = f"| {i} | {X[i]:.5f} | {Y[i]:.6f} | "
        for j in range(1, n + 1 - i):
            row += f"{diff_table[i][j]:.6f} | "
        for j in range(n + 1 - i, n + 1):
            row += " | "
        md.append(row)
    
    display(Markdown('\n'.join(md)))
    
    # Quyết định dùng Newton Tiến hay Lùi
    mid_point = (X[0] + X[-1]) / 2
    
    if t_val <= mid_point:
        method = "Tiến (Forward)"
        # Tìm x_0 gần t_val nhất mà t_val > x_0
        idx = 0
        for i in range(len(X)-degree):
            if X[i] <= t_val < X[i+1]:
                idx = i
                break
        
        x0 = X[idx]
        y0 = Y[idx]
        s = (t_val - x0) / h
        
        display(Markdown(f"\n### ❖ NỘI SUY NEWTON TIẾN"))
        display(Markdown(f"Điểm cần tính $t = {t_val}$ nằm ở nửa đầu bảng. Chọn mốc $x_0 = x_{{{idx}}} = {x0}$."))
        display(Math(f"h = {h:.5f}, \\quad s = \\frac{{t - x_0}}{{h}} = \\frac{{{t_val} - {x0}}}{{{h}}} = {s:.5f}"))
        
        # Giá trị nội suy
        val = y0
        term_s = 1.0
        
        eq_str = f"P_{{{degree}}}({t_val}) = {y0:.6f}"
        
        # Đạo hàm
        deriv = 0.0
        
        for i in range(1, degree + 1):
            delta_y = diff_table[idx][i]
            
            # Tính s(s-1)...(s-i+1)
            term_s *= (s - i + 1)
            val += (term_s * delta_y) / math.factorial(i)
            
            eq_str += f" + \\frac{{s(s-1)...(s-{i-1})}}{{{i}!}} ({delta_y:.6f})"
            
            # Đạo hàm dP/ds (cách đơn giản nhất là xấp xỉ số hoặc tính giải tích các hệ số đầu tiên)
            # Vì đề thi yêu cầu tính, ta có công thức đạo hàm tại x = x0 + sh:
            # P'(x) = 1/h * [ d y0 + (2s-1)/2 d2 y0 + (3s^2-6s+2)/6 d3 y0 + ... ]
        
        # Tính chính xác đạo hàm bằng sai phân tiến tới bậc 5
        dP_ds = 0
        if degree >= 1: dP_ds += diff_table[idx][1]
        if degree >= 2: dP_ds += diff_table[idx][2] * (2*s - 1) / 2.0
        if degree >= 3: dP_ds += diff_table[idx][3] * (3*s**2 - 6*s + 2) / 6.0
        if degree >= 4: dP_ds += diff_table[idx][4] * (4*s**3 - 18*s**2 + 22*s - 6) / 24.0
        if degree >= 5: dP_ds += diff_table[idx][5] * (5*s**4 - 40*s**3 + 105*s**2 - 100*s + 24) / 120.0
        
        deriv = dP_ds / h
        
        display(Math(eq_str))
        display(Math(f"\\Rightarrow P_{{{degree}}}({t_val}) \\approx {val:.6f}"))
        
        display(Markdown(f"**Đạo hàm tại $t = {t_val}$:**"))
        display(Math(f"P'_{{{degree}}}({t_val}) = \\frac{{1}}{{h}} \\frac{{dP}}{{ds}} \\approx {deriv:.6f}"))
        
    else:
        method = "Lùi (Backward)"
        # Tìm x_n
        idx = len(X) - 1
        for i in range(degree, len(X)):
            if X[i-1] < t_val <= X[i]:
                idx = i
                break
                
        xn = X[idx]
        yn = Y[idx]
        s = (t_val - xn) / h
        
        display(Markdown(f"\n### ❖ NỘI SUY NEWTON LÙI"))
        display(Markdown(f"Điểm cần tính $t = {t_val}$ nằm ở nửa sau bảng. Chọn mốc $x_n = x_{{{idx}}} = {xn}$."))
        display(Math(f"h = {h:.5f}, \\quad s = \\frac{{t - x_n}}{{h}} = \\frac{{{t_val} - {xn}}}{{{h}}} = {s:.5f}"))
        
        val = yn
        term_s = 1.0
        eq_str = f"P_{{{degree}}}({t_val}) = {yn:.6f}"
        
        for i in range(1, degree + 1):
            nabla_y = diff_table[idx-i][i] # nabla^i y_n = delta^i y_{n-i}
            term_s *= (s + i - 1)
            val += (term_s * nabla_y) / math.factorial(i)
            eq_str += f" + \\frac{{s(s+1)...(s+{i-1})}}{{{i}!}} ({nabla_y:.6f})"
            
        # Đạo hàm
        dP_ds = 0
        if degree >= 1: dP_ds += diff_table[idx-1][1]
        if degree >= 2: dP_ds += diff_table[idx-2][2] * (2*s + 1) / 2.0
        if degree >= 3: dP_ds += diff_table[idx-3][3] * (3*s**2 + 6*s + 2) / 6.0
        if degree >= 4: dP_ds += diff_table[idx-4][4] * (4*s**3 + 18*s**2 + 22*s + 6) / 24.0
        if degree >= 5: dP_ds += diff_table[idx-5][5] * (5*s**4 + 40*s**3 + 105*s**2 + 100*s + 24) / 120.0
        
        deriv = dP_ds / h
        
        display(Math(eq_str))
        display(Math(f"\\Rightarrow P_{{{degree}}}({t_val}) \\approx {val:.6f}"))
        
        display(Markdown(f"**Đạo hàm tại $t = {t_val}$:**"))
        display(Math(f"P'_{{{degree}}}({t_val}) = \\frac{{1}}{{h}} \\frac{{dP}}{{ds}} \\approx {deriv:.6f}"))

# DỮ LIỆU ĐỀ BÀI (Câu 1.1)
# Bảng giá trị Gamma trên [1; 2] với h = 0.05
X_data = np.arange(1.0, 2.01, 0.05)
Y_data = [1.00000, 0.97350, 0.95135, 0.93304, 0.91817, 0.90640, 0.89747, 0.89115, 
          0.88726, 0.88565, 0.88623, 0.88887, 0.89352, 0.90012, 0.90864, 0.91906, 
          0.93138, 0.94561, 0.96177, 0.97988, 1.00000]

# Nội suy bậc 5 tại t = 1.57
Newton_Interpolation(X_data, Y_data, t_val=1.57, degree=5)

# ═══════════════════════════════════════════════════════════════════
# NHẬP DỮ LIỆU CỦA BẠN VÀO ĐÂY
# ═══════════════════════════════════════════════════════════════════
# Bang so lieu (x_i, f(x_i))
x_nodes = [1.0, 2.0, 3.0, 4.0, 5.0]
f_nodes = [1.0, 8.0, 27.0, 64.0, 125.0]  # Vi du: f(x) = x^3

x_target = 2.5    # gia tri can noi suy
# ═══════════════════════════════════════════════════════════════════

Newton_Interpolation(x_nodes, f_nodes, x_target)
